Integrating all code snipppets to get one script as I lost the overview what is up to date to be honest .. to many


I've run the naming of the clustering (TF-IDF) and the clustering with the answers again after handling the noise because I think we forgot that in the parted script before :)

In [25]:
# Imports
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import TSNE
import umap
import hdbscan
import random
import torch
import collections
import copy
from time import sleep

In [53]:
%pip install mistralai

from mistralai import Mistral
api_key = "ujI60UR6Fe5jel48SAtfnMiN5Skxfwhq"
modelAI = "open-mistral-nemo"
client = Mistral(api_key=api_key)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


Load the competency questions for 200 documents and generate clusters with HBDSCAN using only CQs

In [39]:
# Load competency questions from file
with open("competency_questions_output/competency_questions_200_documents.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Extract questions and sources as pairs
pattern = r'\*\*Frage:\*\*\s*(.+?)\s*\*\*Quelle:\*\*\s*(.+?)(?=\n\d+\.\Z|\n\d+\.|\Z)'
entries = re.findall(pattern, content, re.DOTALL)

# Separate question and source for processing
questions_only = [q.strip() for q, s in entries]
sources_only = [s.strip() for q, s in entries]

# Deduplicate questions (with mapping)
seen = set()
questions, sources = [], []
for q, s in zip(questions_only, sources_only):
    if q not in seen:
        seen.add(q)
        questions.append(q)
        sources.append(s)

In [19]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
# Sentence Embedding (SBERT)
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = model.encode(questions)

# Dimensionality Reduction (UMAP, optional but good for HDBSCAN & visualization)
umap_reducer = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
X_umap = umap_reducer.fit_transform(embeddings)

# HDBSCAN clustering (as in the Ntropy article)
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=1, metric='euclidean', prediction_data=True)
labels = clusterer.fit_predict(X_umap)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f"📌 Number of clusters found by HDBSCAN: {n_clusters}")
# Evaluation (excluding outliers)
valid_mask = labels != -1
sil_score = silhouette_score(X_umap[valid_mask], labels[valid_mask])
print(f"📈 Silhouette Score (without outliers): {sil_score:.4f}")

# Visualization (t-SNE)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_2d = tsne.fit_transform(X_umap)

plt.figure(figsize=(10, 6))
palette = sns.color_palette('hls', n_clusters)
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=labels, palette=palette)
plt.title("t-SNE visualization of clustered questions")
plt.show()

# Save basic Cluster 
clustered_cqs = {}
for q, l, e in zip(questions, labels, embeddings):
    entry = {"cq": q, "embedding": e.tolist()}
    clustered_cqs.setdefault(f"Cluster {l}", []).append(entry)

with open("competency_questions_output/clustered_CQs_with_embeddings_lj.json", "w", encoding="utf-8") as f:
    json.dump(clustered_cqs, f, ensure_ascii=False, indent=4)

/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


📌 Number of clusters found by HDBSCAN: 139


/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Use noise triage to handle noise in clusters based only on CQs

17.98% percent of identified noise is still kept after reviewing noise

In [22]:
#Handle noise (triage) 
data = clustered_cqs
noise_cl = data.get("Cluster -1", [])
print(f"Noise CQs: {len(noise_cl)} / {sum(len(v) for v in data.values())} total")

PROMOTE_THRESHOLD = 0.2
REVIEW_THRESHOLD = 0.25

noise_embeddings = np.array([x["embedding"] for x in noise_cl])
core_embeddings = []
core_index_to_cluster = {}
c_idx = 0
for cname, cqs in data.items():
    if cname == "Cluster -1": continue
    for cq in cqs:
        core_embeddings.append(cq["embedding"])
        core_index_to_cluster[c_idx] = cname
        c_idx += 1
core_embeddings = np.array(core_embeddings)

from sklearn.metrics.pairwise import cosine_distances
dist_matrix = cosine_distances(noise_embeddings, core_embeddings)
d_min = dist_matrix.min(axis=1)
nearest_idx = dist_matrix.argmin(axis=1)

triaged = {"promote": [], "review": [], "leave": []}
for i, cq in enumerate(noise_cl):
    distance = d_min[i]
    nearest_cluster = core_index_to_cluster[nearest_idx[i]]
    tag = "promote" if distance <= PROMOTE_THRESHOLD else "review" if distance <= REVIEW_THRESHOLD else "leave"
    triaged[tag].append({"cq": cq["cq"], "embedding": cq["embedding"], "d_min": float(distance), "nearest_cluster": nearest_cluster})

with open("competency_questions_output/noise_triage_lj.json", "w", encoding="utf-8") as f:
    json.dump(triaged, f, ensure_ascii=False, indent=2)


updated_data = {k: copy.deepcopy(v) for k, v in data.items() if k != "Cluster -1"}
for tag in ["promote", "review"]:
    for item in triaged[tag]:
        item["from_noise"] = tag
        updated_data[item["nearest_cluster"]].append(item)
updated_data["Cluster -1"] = [{**item, "from_noise": "leave"} for item in triaged["leave"]]

ordered_data = dict(sorted(updated_data.items(), key=lambda x: (x[0] != "Cluster -1", int(x[0].split()[-1]) if x[0].startswith("Cluster ") else float('inf'))))

with open("competency_questions_output/clustered_CQs_with_promoted_and_reviewed_noise_lj.json", "w", encoding="utf-8") as f:
    json.dump(ordered_data, f, ensure_ascii=False, indent=2)


Noise CQs: 307 / 1709 total


Use TF-IDF to label the noise-handled clusters (based only on CQs)

In [ ]:
#TF_IDF labeling and evaluation
stopwords_de = ["aber", "alle", "als", "also", "am", "an", "ander", "auch", "auf", "aus", "bei", "bin", "bis", "bist", "da", "dadurch", "daher", "darum", "das", "dass", "dein", "deine", "dem", "den", "der", "des", "dessen", "deshalb", "die", "dies", "dieser", "dieses", "doch", "dort", "du", "durch", "ein", "eine", "einem", "einen", "einer", "eines", "er", "es", "euer", "eure", "für", "hatte", "hatten", "hattest", "hattet", "hier", "hinter", "ich", "ihr", "ihre", "im", "in", "ist", "ja", "jede", "jedem", "jeden", "jeder", "jedes", "jener", "jenes", "jetzt", "kann", "kannst", "können", "könnt", "machen", "mein", "meine", "mit", "muß", "mußt", "musst", "müssen", "müßt", "nach", "nachdem", "nein", "nicht", "nun", "oder", "seid", "sein", "seine", "sich", "sie", "sind", "soll", "sollen", "sollst", "sollt", "sonst", "soweit", "sowie", "und", "unser", "unsere", "unter", "vom", "von", "vor", "wann", "warum", "was", "weiter", "weitere", "wenn", "wer", "werde", "werden", "werdet", "weshalb", "wie", "wieder", "wir", "wird", "wirst", "wo", "woher", "wohin", "zu", "zum", "zur", "über"]
synonym_map = {"vergütung": ["entlohnung", "honorar", "bezahlung"], "kündigung": ["beendigung", "auflösung"], "vertrag": ["vereinbarung"]}

def apply_synonyms(texts, mapping):
    for t_idx in range(len(texts)):
        for target, syns in mapping.items():
            for s in syns:
                texts[t_idx] = texts[t_idx].replace(s, target)
    return texts

all_questions, all_labels = [], []
for cname, cqs in ordered_data.items():
    if cname == "Cluster -1": continue
    for cq in cqs:
        all_questions.append(cq["cq"])
        all_labels.append(cname)

questions_syn = apply_synonyms(list(all_questions), synonym_map)
vectorizer = TfidfVectorizer(stop_words=stopwords_de)
X_tfidf = vectorizer.fit_transform(questions_syn)

feature_names = np.array(vectorizer.get_feature_names_out())
cluster_to_questions = collections.defaultdict(list)
for question, label in zip(questions_syn, all_labels):
    cluster_to_questions[label].append(question)

print("\n\U0001F50E Top TF-IDF Keywords per Cluster:")
for cluster_id, texts in cluster_to_questions.items():
    matrix = vectorizer.transform(texts).mean(axis=0).A1
    top_keywords = feature_names[matrix.argsort()[-7:][::-1]]
    print(f"\n\U0001F4C1 {cluster_id} (n={len(texts)}):\nTop Keywords: {', '.join(top_keywords)}")
    



🔎 Top TF-IDF Keywords per Cluster:

📁 Cluster 0 (n=8):
Top Keywords: netz, kabelnetzbetreiber, betreibers, anderen, signaldurchleitung, signale, kabelnetzes

📁 Cluster 1 (n=9):
Top Keywords: patentnichtigkeitsverfahren, kläger, patentnichtigkeitsklage, patent, rechtsschutzmöglichkeit, angegriffene, argumentieren

📁 Cluster 2 (n=8):
Top Keywords: unerlaubter, handlung, thailand, ansprüchen, verjährung, thailändischem, recht

📁 Cluster 3 (n=10):
Top Keywords: entschädigung, reise, bemessung, ersatzreise, vereitelt, vereitelter, höhe

📁 Cluster 4 (n=5):
Top Keywords: geschäftsbedingungen, allgemeinen, reisende, hat, reisevertrag, reisender, kenntnisnahme

📁 Cluster 5 (n=8):
Top Keywords: reisende, reise, ansprüche, reiseveranstalter, reisenden, verjähren, verjährungsfrist

📁 Cluster 6 (n=9):
Top Keywords: geräuschemissionen, zumutbarkeitsgrenze, schallschutzstandard, mietwohnung, 906, bestimmen, bestimmung

📁 Cluster 7 (n=6):
Top Keywords: abtretungsklausel, transparenz, verständlichkeit

Generate generalized questions for the noise reduced clusters (clusters based only on CQs)

In [ ]:
results = {}

system_prompt = """
You are a legal knowledge engineering expert.

Your task is to generate a **generalized competency question (GCQ)** from a list of **clustered German legal competency questions (CQs)**.

---

## Goal:
Create **one single abstract, representative legal question** that captures the core meaning of all input CQs.

## Instructions:
- Output one well-formed, precise, **legally relevant** question in German.
- The question should reflect the **shared semantics** of the cluster without copying specific details.
- Use **legal terminology** such as:
  - “Unter welchen Voraussetzungen…”
  - “Welche rechtliche Bedeutung hat…”
  - “Wer ist verpflichtet…”
- Do not summarize, explain, or list individual CQs. Just output the **generalized question**.
- Output **only the question**, in natural, legal German.
"""

for cluster_id, questions in ordered_data.items():
    prompt = system_prompt + "\n\nFragen:\n" + "\n".join(f"- {q['cq']}" for q in questions if 'cq' in q) + "\n\nGeneralisierte Frage:"
    try:
        response = client.chat.complete(
            model=modelAI,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )
        results[cluster_id] = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Fehler bei Cluster {cluster_id}:", e)
        results[cluster_id] = "Error generating question."
    sleep(5)

with open("competency_questions_output/generalized_questions_lj.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("Fertig – generalisierte Fragen wurden gespeichert.")


Fertig – generalisierte Fragen wurden gespeichert.


Repeat the process but include the sources of the questions for clustering

In [45]:
# Combine question and source into a single string for embedding
combined_texts = [f"Frage: {q} Kontext: {s}" for q, s in zip(questions, sources)]
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
combined_embeddings = model.encode(combined_texts)

# Dimensionality reduction - adapt the dimensions 
umap_reducer_combined = umap.UMAP(n_neighbors=15, n_components=5, metric='cosine', random_state=42)
X_umap_combined = umap_reducer_combined.fit_transform(combined_embeddings)

# HDBSCAN clustering
clusterer_combined = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=1, metric='euclidean', prediction_data=True)
labels_combined = clusterer_combined.fit_predict(X_umap_combined)
n_clusters_combined = len(set(labels_combined) - {-1})
print(f"📌 Number of clusters (combined CQ + source): {n_clusters_combined}")

# Save combined clusters
clustered_combined = {}
for question, source, label, emb in zip(questions, sources, labels_combined, combined_embeddings):
    entry = {"cq": question, "quelle": source, "embedding": emb.tolist()}
    clustered_combined.setdefault(f"Cluster {label}", []).append(entry)

with open("competency_questions_output/clustered_CQs_combined_lj.json", "w", encoding="utf-8") as f:
    json.dump(clustered_combined, f, ensure_ascii=False, indent=4)

/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Applications/anaconda3/envs/cq_env/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


📌 Number of clusters (combined CQ + source): 129


Use noise triage and update the clusters (CQs + sources)
ca. 19.6% are kept as noise

In [47]:
# --- 13. Triage Noise from Combined Clustering ---
noise_combined = clustered_combined.get("Cluster -1", [])
print(f"Noise from combined: {len(noise_combined)} / {sum(len(v) for v in clustered_combined.values())} total")

# Prepare core embeddings
core_combined_embeddings = []
core_combined_map = {}
idx = 0
for cname, cqs in clustered_combined.items():
    if cname == "Cluster -1": continue
    for cq in cqs:
        core_combined_embeddings.append(cq["embedding"])
        core_combined_map[idx] = cname
        idx += 1
core_combined_embeddings = np.array(core_combined_embeddings)
noise_combined_embeddings = np.array([cq["embedding"] for cq in noise_combined])

# Cosine distance
dist_combined = cosine_distances(noise_combined_embeddings, core_combined_embeddings)
d_min_combined = dist_combined.min(axis=1)
nearest_idx_combined = dist_combined.argmin(axis=1)

triaged_combined = {"promote": [], "review": [], "leave": []}
for i, cq in enumerate(noise_combined):
    dist = d_min_combined[i]
    nearest_cluster = core_combined_map[nearest_idx_combined[i]]
    tag = "promote" if dist <= PROMOTE_THRESHOLD else "review" if dist <= REVIEW_THRESHOLD else "leave"
    triaged_combined[tag].append({"cq": cq["cq"], "quelle": cq["quelle"], "embedding": cq["embedding"], "d_min": float(dist), "nearest_cluster": nearest_cluster})

with open("competency_questions_output/noise_triage_combined_lj.json", "w", encoding="utf-8") as f:
    json.dump(triaged_combined, f, ensure_ascii=False, indent=2)

# --- 14. Updated clusters after combined triage ---
updated_combined_clusters = {k: copy.deepcopy(v) for k, v in clustered_combined.items() if k != "Cluster -1"}
for tag in ["promote", "review"]:
    for item in triaged_combined[tag]:
        item["from_noise"] = tag
        updated_combined_clusters[item["nearest_cluster"]].append(item)
updated_combined_clusters["Cluster -1"] = [{**item, "from_noise": "leave"} for item in triaged_combined["leave"]]

with open("competency_questions_output/clustered_CQs_combined_with_noise_handled_lj.json", "w", encoding="utf-8") as f:
    json.dump(updated_combined_clusters, f, ensure_ascii=False, indent=2)

print("\n✅ Combined clustering and noise triage completed. Output saved.")

Noise from combined: 336 / 1709 total

✅ Combined clustering and noise triage completed. Output saved.


Use TF-IDF to label the noise-handled clusters (based on CQs + sources)

In [49]:
# --- 15. TF-IDF Labeling for Combined Clusters ---
stopwords_de = ["aber", "alle", "als", "also", "am", "an", "ander", "auch", "auf", "aus", "bei", "bin", "bis", "bist", "da", "dadurch", "daher", "darum", "das", "dass", "dein", "deine", "dem", "den", "der", "des", "dessen", "deshalb", "die", "dies", "dieser", "dieses", "doch", "dort", "du", "durch", "ein", "eine", "einem", "einen", "einer", "eines", "er", "es", "euer", "eure", "für", "hatte", "hatten", "hattest", "hattet", "hier", "hinter", "ich", "ihr", "ihre", "im", "in", "ist", "ja", "jede", "jedem", "jeden", "jeder", "jedes", "jener", "jenes", "jetzt", "kann", "kannst", "können", "könnt", "machen", "mein", "meine", "mit", "muß", "mußt", "musst", "müssen", "müßt", "nach", "nachdem", "nein", "nicht", "nun", "oder", "seid", "sein", "seine", "sich", "sie", "sind", "soll", "sollen", "sollst", "sollt", "sonst", "soweit", "sowie", "und", "unser", "unsere", "unter", "vom", "von", "vor", "wann", "warum", "was", "weiter", "weitere", "wenn", "wer", "werde", "werden", "werdet", "weshalb", "wie", "wieder", "wir", "wird", "wirst", "wo", "woher", "wohin", "zu", "zum", "zur", "über"]
synonym_map = {"vergütung": ["entlohnung", "honorar", "bezahlung"], "kündigung": ["beendigung", "auflösung"], "vertrag": ["vereinbarung"]}

def apply_synonyms(texts, mapping):
    for t_idx in range(len(texts)):
        for target, syns in mapping.items():
            for s in syns:
                texts[t_idx] = texts[t_idx].replace(s, target)
    return texts

combined_questions, combined_labels = [], []
for cname, cqs in updated_combined_clusters.items():
    if cname == "Cluster -1": continue
    for cq in cqs:
        combined_questions.append(cq["cq"])
        combined_labels.append(cname)

combined_questions_syn = apply_synonyms(list(combined_questions), synonym_map)
vectorizer_combined = TfidfVectorizer(stop_words=stopwords_de)
X_tfidf_combined = vectorizer_combined.fit_transform(combined_questions_syn)

feature_names_combined = np.array(vectorizer_combined.get_feature_names_out())
cluster_to_questions_combined = collections.defaultdict(list)
for question, label in zip(combined_questions_syn, combined_labels):
    cluster_to_questions_combined[label].append(question)

print("\n🔍 Top TF-IDF Keywords per Combined Cluster:")
for cluster_id, texts in cluster_to_questions_combined.items():
    matrix = vectorizer_combined.transform(texts).mean(axis=0).A1
    top_keywords = feature_names_combined[matrix.argsort()[-7:][::-1]]
    print(f"\n📁 {cluster_id} (n={len(texts)}):\nTop Keywords: {', '.join(top_keywords)}")



🔍 Top TF-IDF Keywords per Combined Cluster:

📁 Cluster 28 (n=6):
Top Keywords: grundbuch, rück, eigentümer, auflassung, eingetragener, verlangen, grundbuchberichtigung

📁 Cluster 29 (n=10):
Top Keywords: eigentümer, grundstück, eigentum, verloren, hat, aufgrund, übertragung

📁 Cluster 26 (n=33):
Top Keywords: grundstücks, überbau, grundstück, errichtet, baulichkeiten, mieter, welche

📁 Cluster 32 (n=31):
Top Keywords: klausel, 57a, zvg, bgb, welche, mietvertrages, abs

📁 Cluster 109 (n=7):
Top Keywords: durchführt, recht, gericht, ansprüchen, schädigungsvorsatz, italienischen, niederländische

📁 Cluster 60 (n=14):
Top Keywords: vertragshändler, auftraggeber, klausel, hat, erfüllt, welche, informationspflichten

📁 Cluster 114 (n=15):
Top Keywords: 281, verjährungsfrist, bgb, fristsetzung, abs, dauert, frist

📁 Cluster 100 (n=9):
Top Keywords: inhaltskontrolle, geschäftsbedingungen, allgemeinen, ff, klausel, bgb, unwirksam

📁 Cluster 110 (n=15):
Top Keywords: ordnungsgemäß, rechtsfolge,

Generate generalized questions based on clusters from CQs + source

In [54]:
combined_gcq_results = {}

for cluster_id, questions in updated_combined_clusters.items():
    prompt = (
        system_prompt
        + "\n\nFragen:\n"
        + "\n".join(f"- {q['cq']}" for q in questions if 'cq' in q)
        + "\n\nGeneralisierte Frage:"
    )
    try:
        response = client.chat.complete(
            model=modelAI,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ]
        )
        combined_gcq_results[cluster_id] = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Fehler bei Cluster {cluster_id}:", e)
        combined_gcq_results[cluster_id] = "Error generating question."
    sleep(5)

with open("competency_questions_output/generalized_questions_combined_lj.json", "w", encoding="utf-8") as f:
    json.dump(combined_gcq_results, f, indent=2, ensure_ascii=False)

print("✅ Generalized questions for combined clustering saved.")

✅ Generalized questions for combined clustering saved.


Compare both clustering methods (only CQs  vs. CQs + source)

If I did no mistake, the clusters only based on CQs have overall slightly better metric scores than the ones that include the source

In [51]:
import json

with open("competency_questions_output/clustered_CQs_with_promoted_and_reviewed_noise_lj.json", "r", encoding="utf-8") as f:
    clusters_cq_only = json.load(f)

with open("competency_questions_output/clustered_CQs_combined_with_noise_handled_lj.json", "r", encoding="utf-8") as f:
    clusters_combined = json.load(f)

num_clusters_cq_only = sum(1 for k in clusters_cq_only if k != "Cluster -1")
num_clusters_combined = sum(1 for k in clusters_combined if k != "Cluster -1")

print(f"📊 Clusters using CQs only: {num_clusters_cq_only}")
print(f"📊 Clusters using CQ + Source: {num_clusters_combined}")


def evaluate_clusters(data_dict, label):
    emb_list, lbl_list = [], []
    cluster_map = {}
    counter = 0
    for cname, cqs in data_dict.items():
        if cname == "Cluster -1": continue
        if cname not in cluster_map:
            cluster_map[cname] = counter
            counter += 1
        for cq in cqs:
            emb_list.append(cq["embedding"])
            lbl_list.append(cluster_map[cname])
    emb_list = np.array(emb_list)
    lbl_list = np.array(lbl_list)
    sil = silhouette_score(emb_list, lbl_list, metric="cosine")
    db = davies_bouldin_score(emb_list, lbl_list)
    ch = calinski_harabasz_score(emb_list, lbl_list)
    print(f"\n📊 {label}:")
    print(f"Silhouette Score:        {sil:.4f}")
    print(f"Davies-Bouldin Score:    {db:.4f}")
    print(f"Calinski-Harabasz Score: {ch:.2f}")

# Run evaluation for both
evaluate_clusters(ordered_data, "Evaluation – Only CQs")
evaluate_clusters(updated_combined_clusters, "Evaluation – Combined CQ + Source")


📊 Clusters using CQs only: 139
📊 Clusters using CQ + Source: 129

📊 Evaluation – Only CQs:
Silhouette Score:        0.1252
Davies-Bouldin Score:    2.2174
Calinski-Harabasz Score: 13.58

📊 Evaluation – Combined CQ + Source:
Silhouette Score:        0.0998
Davies-Bouldin Score:    2.3429
Calinski-Harabasz Score: 10.98
